# Answer Synthesis: Reconcile Conflicts with Provenance

| Field | Value |
|---|---|
| Stage | Autonomous RAG patterns |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Synthesis is not concatenation. Conflicting evidence needs scope, effective dates, and a declared resolution rule.

## 30-Second Summary

This notebook merges two retention policies with different effective dates. For an as-of date after the migration, the newer 45-day policy controls; both sources remain visible and the conflict is disclosed.

## Why This Matters

Parallel retrieval can return contradictory facts. A fluent merger that hides disagreement creates false certainty and breaks citations.

## Scope

| Covers | Does not cover |
|---|---|
| Provenance records, effective-date rule, conflict detection, cited synthesis | Legal interpretation, semantic contradiction model, arbitrary source authority |


## Mental Model

```text
evidence records -> normalize field/scope/date -> detect conflict -> resolve by rule -> answer + citations + caveat
```


In [1]:
from datetime import date

evidence = [
    {"source": "policy-v1", "field": "retention_days", "value": 30, "effective": date(2025, 1, 1)},
    {"source": "migration-v2", "field": "retention_days", "value": 45, "effective": date(2026, 10, 1)},
]
as_of = date(2026, 11, 1)


## How It Works

Records share a field and scope, so differing values are a conflict. The declared rule selects the latest policy effective on or before `as_of`; the answer cites the controlling source and names the superseded value.


## Baseline

Naive concatenation reports both values without explaining which applies.


In [2]:
baseline = "Logs are retained for 30 days [policy-v1] and 45 days [migration-v2]."
baseline


'Logs are retained for 30 days [policy-v1] and 45 days [migration-v2].'

## Technique Implementation

The resolver filters out future records, sorts by effective date, detects distinct values, and retains every source for audit.


In [3]:
applicable = sorted((item for item in evidence if item["effective"] <= as_of), key=lambda item: item["effective"])
controlling = applicable[-1]
conflict = len({item["value"] for item in applicable}) > 1
resolution = {
    "value": controlling["value"],
    "source": controlling["source"],
    "conflict": conflict,
    "superseded": [item for item in applicable[:-1] if item["value"] != controlling["value"]],
}
resolution


{'value': 45,
 'source': 'migration-v2',
 'conflict': True,
 'superseded': [{'source': 'policy-v1',
   'field': 'retention_days',
   'value': 30,
   'effective': datetime.date(2025, 1, 1)}]}

## Controlled Experiment

We synthesize for two as-of dates—before and after migration—to verify the boundary and preserve citations.


In [4]:
def resolve(as_of_date: date) -> dict:
    candidates = sorted((item for item in evidence if item["effective"] <= as_of_date), key=lambda item: item["effective"])
    if not candidates: return {"answer": None, "source": None}
    chosen = candidates[-1]
    return {
        "answer": f"As of {as_of_date.isoformat()}, retention is {chosen['value']} days [{chosen['source']}].",
        "source": chosen["source"],
    }

before = resolve(date(2026, 9, 30))
after = resolve(as_of)
results = {"before": before, "after": after, "conflict_detected": conflict}
results


{'before': {'answer': 'As of 2026-09-30, retention is 30 days [policy-v1].',
  'source': 'policy-v1'},
 'after': {'answer': 'As of 2026-11-01, retention is 45 days [migration-v2].',
  'source': 'migration-v2'},
 'conflict_detected': True}

## Evaluation

Before 2026-10-01, the 30-day policy controls. After it, the 45-day migration policy controls. The conflict is detected rather than averaged or hidden, and each answer cites its controlling source.


In [5]:
assert results["before"]["source"] == "policy-v1" and "30 days" in results["before"]["answer"]
assert results["after"]["source"] == "migration-v2" and "45 days" in results["after"]["answer"]
assert results["conflict_detected"] and resolution["superseded"][0]["source"] == "policy-v1"
print("Answer-synthesis conflict checks passed.")


Answer-synthesis conflict checks passed.


## Decision Guide

| Evidence state | Synthesis |
|---|---|
| Consistent facts | Merge with citations |
| Versioned conflict | Apply declared effective-date/authority rule |
| Unresolved authority | Surface disagreement |
| Missing required fact | Partial answer or abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Two values silently averaged | Numeric merge without semantics | Field-specific conflict rule |
| Old policy cited as current | Dates ignored | As-of filtering |
| Caveat lost | Provenance stripped | Structured evidence through synthesis |
| New source always wins | Recency mistaken for authority | Source ownership hierarchy |


## Production Notes

### Observability
Log evidence IDs, normalized fields/units, conflicts, resolution rule/version, as-of date, and citations.

### Safety and Guardrails
Never merge evidence across unauthorized tenants or scopes.

### Latency and Cost
Resolve deterministic metadata conflicts before invoking a model.


## Practice

Add a future-dated 60-day policy and prove it is ignored before its effective date.

## Recall

Toggle - Recall: What makes synthesis trustworthy?
Structured provenance, explicit conflict rules, and citations.

Toggle - Recall: Should conflicting values be averaged?
Only if the metric definition explicitly supports aggregation; policy values generally should not be.

## Sources

- [LangGraph parallelization and synthesis](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic versioned policies

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the effective-date rule fixture | Add authority hierarchies and unresolved conflicts |
